# Rotten Tomatoes Sentiment Analysis

## Environment Setup

First, let's import the necessary libraries and set up our environment.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from collections import Counter
from tqdm import tqdm
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# Download required NLTK resources
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('stopwords')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Data Loading

Let's load and examine the Rotten Tomatoes dataset.

In [ ]:
# Load the train and test data
train_df = pd.read_csv('./train.tsv', sep='\t')
test_df = pd.read_csv('./test.tsv', sep='\t')

print("Training data shape:", train_df.shape)
print("Test data shape:", test_df.shape)
train_df.head()

## Data Analysis and Preprocessing

In [ ]:
# Drop non-relevant columns and missing values
train_df = train_df.dropna()
train_df = train_df.drop(['PhraseId', 'SentenceId'], axis=1)

# Look at the distribution of sentiment labels
plt.figure(figsize=(10, 6))
train_df['Sentiment'].value_counts().sort_index().plot(kind='bar', color='skyblue')
plt.title('Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Number of Phrases')
plt.xticks(ticks=[0, 1, 2, 3, 4], labels=['Very Negative', 'Negative', 'Neutral', 'Positive', 'Very Positive'], rotation=45)
plt.grid(axis='y')
plt.show()

### Text Cleaning Function

In [ ]:
# Set up stopwords and punctuation
stop_words = set(stopwords.words('english'))
punctuation = set(string.punctuation)
lemmatizer = nltk.stem.WordNetLemmatizer()

def clean_text(text):
    tokens = word_tokenize(text.lower())

    cleaned = []
    for token in tokens:
        if token in stop_words or token in punctuation:
            continue
        if token.isnumeric():
            continue
        if re.match(r'^[a-zA-Z]+$', token):
            lemmatized = lemmatizer.lemmatize(token)
            cleaned.append(lemmatized)

    return ' '.join(cleaned)

# Apply cleaning to the phrases
train_df['Cleaned_Phrase'] = train_df['Phrase'].apply(clean_text)

# Remove empty strings
train_df = train_df[train_df['Cleaned_Phrase'].str.strip() != '']

# Display some examples
print(train_df[['Phrase', 'Cleaned_Phrase']].sample(5))

### Compare Phrase Lengths

In [ ]:
# Compare original and cleaned phrase lengths
train_df['Phrase_Length'] = train_df['Phrase'].apply(lambda x: len(x.split()))
train_df['Cleaned_Phrase_Length'] = train_df['Cleaned_Phrase'].apply(lambda x: len(x.split()))

plt.figure(figsize=(10, 6))
plt.hist(train_df['Phrase_Length'], bins=50, alpha=0.5, label='Original Phrase Length')
plt.hist(train_df['Cleaned_Phrase_Length'], bins=50, alpha=0.5, label='Cleaned Phrase Length')
plt.title('Comparison of Phrase Length Distributions')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.legend()
plt.show()

# Keep only necessary columns
train_df = train_df.drop(['Phrase', 'Phrase_Length', 'Cleaned_Phrase_Length'], axis=1)
train_df = train_df.reset_index(drop=True)
train_df.head()

## Building the Vocabulary

In [ ]:
def build_vocab(phrases, min_freq=1, max_vocab_size=None):
    counter = Counter(word for phrase in phrases for word in phrase.split())

    vocab = {"<pad>": 0, "<unk>": 1}

    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
            if max_vocab_size and len(vocab) >= max_vocab_size:
                break

    return vocab

## Dataset and Data Loading Utilities

In [ ]:
class PhraseDataset(Dataset):
    def __init__(self, phrases, labels, word2idx, max_length=None):
        self.phrases = phrases
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.word2idx = word2idx
        self.max_length = max_length

    def __len__(self):
        return len(self.phrases)

    def __getitem__(self, idx):
        phrase = self.phrases[idx]
        tokens = [self.word2idx.get(word, self.word2idx.get("<unk>")) for word in phrase.split()]

        encoded_phrase = torch.tensor(tokens, dtype=torch.long)

        return encoded_phrase, self.labels[idx]

def collate_fn(batch):
    batch.sort(key=lambda x: len(x[0]), reverse=True)
    phrases, labels = zip(*batch)
    lengths = torch.tensor([len(phrase) for phrase in phrases])
    padded_phrases = pad_sequence(phrases, batch_first=True, padding_value=0)
    labels = torch.stack(labels)

    return padded_phrases, lengths, labels

## Model Architecture - BiLSTM

In [ ]:
class PhraseClassifier(nn.Module):
    def __init__(self, vocab_sz, embedding_dim, hidden_sz, num_lstm_layers, dropout_prob, num_classes):
        super(PhraseClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_sz, embedding_dim)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_sz,
            num_layers=num_lstm_layers,
            dropout=dropout_prob if num_lstm_layers > 1 else 0,
            bidirectional=True,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout_prob)
        self.fc = nn.Linear(hidden_sz * 2, num_classes)

    def forward(self, text, lengths):
        embedded = self.embedding(text)

        packed_embedded = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=True
        )
        packed_output, (hidden, _) = self.lstm(packed_embedded)

        hidden_concat = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        hidden_dropout = self.dropout(hidden_concat)

        return self.fc(hidden_dropout)

## Training Setup

In [ ]:
# Hyperparameters
EMBEDDING_DIM = 100
HIDDEN_SIZE = 128
NUM_LSTM_LAYERS = 2
DROPOUT = 0.2
BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_EPOCHS = 6
NUM_CLASSES = 5

# Training function
def train_model(model, train_loader, valid_loader, criterion, optimizer, device, num_epochs=5):
    best_valid_loss = float('inf')

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        correct_predictions = 0
        total_predictions = 0

        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')

        for batch in progress_bar:
            phrases, lengths, labels = batch
            phrases, labels = phrases.to(device), labels.to(device)

            optimizer.zero_grad()

            predictions = model(phrases, lengths)

            loss = criterion(predictions, labels)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            _, predicted_classes = predictions.max(1)
            correct_predictions += (predicted_classes == labels).sum().item()
            total_predictions += labels.size(0)

            progress_bar.set_postfix({
                'loss': loss.item(),
                'accuracy': correct_predictions / total_predictions
            })

        train_loss = epoch_loss / len(train_loader)
        train_accuracy = correct_predictions / total_predictions

        print(f'Epoch {epoch+1}/{num_epochs}:')
        print(f'\tTrain Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}')

        if valid_loader:
            valid_loss, valid_accuracy = evaluate_model(model, valid_loader, criterion, device)
            print(f'\tValid Loss: {valid_loss:.4f}, Valid Accuracy: {valid_accuracy:.4f}')

            if valid_loss < best_valid_loss:
                best_valid_loss = valid_loss
                torch.save(model.state_dict(), 'best_sentiment_model.pt')
                print(f'\tBest validation loss: {best_valid_loss:.4f}')

        print()

    return model

# Evaluation function
def evaluate_model(model, data_loader, criterion, device):
    model.eval()
    epoch_loss = 0
    correct_predictions = 0
    total_predictions = 0

    with torch.no_grad():
        for batch in data_loader:
            phrases, lengths, labels = batch
            phrases, labels = phrases.to(device), labels.to(device)

            predictions = model(phrases, lengths)

            loss = criterion(predictions, labels)

            epoch_loss += loss.item()

            _, predicted_classes = predictions.max(1)
            correct_predictions += (predicted_classes == labels).sum().item()
            total_predictions += labels.size(0)

    return epoch_loss / len(data_loader), correct_predictions / total_predictions

## Training the Model

In [ ]:
# Split the data into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    train_df["Cleaned_Phrase"].tolist(),
    train_df["Sentiment"].tolist(),
    test_size=0.1,
    random_state=42,
    stratify=train_df["Sentiment"].tolist()
)

# Build the vocabulary
vocab = build_vocab(X_train, min_freq=2)
print(f"Vocabulary size: {len(vocab)}")

# Create datasets
train_dataset = PhraseDataset(X_train, y_train, vocab)
valid_dataset = PhraseDataset(X_val, y_val, vocab)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

# Initialize the model
model = PhraseClassifier(
    vocab_sz=len(vocab),
    embedding_dim=EMBEDDING_DIM,
    hidden_sz=HIDDEN_SIZE,
    num_lstm_layers=NUM_LSTM_LAYERS,
    dropout_prob=DROPOUT,
    num_classes=NUM_CLASSES
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=0.0001)

# Train the model
model = train_model(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=NUM_EPOCHS
)

## Testing and Prediction

In [ ]:
def predict_sentiment(model, vocab, device, phrases):
    model.eval()
    results = []

    for phrase in phrases:
        if not isinstance(phrase, str):
            phrase = ""

        phrase = clean_text(phrase)

        tokens = [vocab.get(word, vocab.get("<unk>")) for word in phrase.split()]
        if len(tokens) == 0:
            tokens = [vocab.get("<pad>")]

        tensor = torch.tensor([tokens], dtype=torch.long).to(device)
        lengths = torch.tensor([len(tokens)])

        with torch.no_grad():
            prediction = model(tensor, lengths)
            _, predicted_class = prediction.max(1)
            sentiment_index = predicted_class.item()

        results.append(sentiment_index)
    return results

# Try on some examples
index_to_sentiment = {
    0: "Very Negative",
    1: "Negative",
    2: "Neutral",
    3: "Positive",
    4: "Very Positive"
}

test_phrases = [
    "this movie was amazing and heartfelt",
    "terrible acting and poor direction",
    "okay"
]

results = predict_sentiment(
    model,
    vocab,
    device,
    test_phrases
)

for i in range(len(results)):
    print(f"Phrase: '{test_phrases[i]}'")
    print(f"Sentiment: {index_to_sentiment[results[i]]}\n")

## Create Kaggle Submission

In [ ]:
# Generate predictions for test data
test_phrases = test_df['Phrase'].fillna("").tolist()

test_predictions = predict_sentiment(model, vocab, device, test_phrases)

test_df['Sentiment'] = test_predictions

output_df = test_df[['PhraseId', 'Sentiment']]

output_df.to_csv('./submission.csv', index=False)
print("Submission file created.")

## Save the Model

In [ ]:
torch.save(model.state_dict(), 'model.pt')
print("Model saved successfully.")